# 07 - BERTopic sobre el subcorpus de salud

In [ ]:
# ============================================================
# CELL 0 - CONFIG
# ============================================================
from pathlib import Path

DATA_PROCESSED = Path(r'C:\Users\afpue\Documents\GitHub\icare\kMetodo\resultadosPropios')

NR_TOPICS_GRID  = [10, 20, 30, 40, 60, "auto"]
NR_TOPICS_FINAL = 20
MIN_TOPIC_SIZE  = 15

print('[CONFIG] OK')
print(f'  DATA_PROCESSED : {DATA_PROCESSED.resolve()}')


In [ ]:
# ============================================================
# CELL 0b - COMPATIBILIDAD NUMPY 2.x / TENSORFLOW
# ============================================================
import sys, types, importlib.machinery

if 'tensorflow' not in sys.modules:
    def _fake_mod(name):
        m = types.ModuleType(name)
        m.__spec__    = importlib.machinery.ModuleSpec(name, loader=None)
        m.__path__    = []
        m.__package__ = name
        m.__version__ = '0.0.0'
        return m
    for _n in [
        'tensorflow', 'tensorflow.python', 'tensorflow.python.eager',
        'tensorflow.python.framework', 'tensorflow.python.client',
        'tensorflow.python.util', 'tensorflow.python.ops',
        'tensorflow.core', 'tensorflow.keras', 'tensorflow.keras.layers',
        'tensorflow.keras.models', 'tensorflow.keras.callbacks',
        'tensorflow.keras.losses', 'tensorflow.keras.optimizers',
        'tensorflow.compat', 'tensorflow.compat.v1', 'tensorflow.compat.v2',
    ]:
        sys.modules[_n] = _fake_mod(_n)
    print('[COMPAT] tensorflow reemplazado con modulos ficticios')
else:
    print('[COMPAT] tensorflow ya estaba cargado, sin cambios')


In [ ]:
# ============================================================
# CELL 1 - IMPORTS Y CARGA
# ============================================================
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

corpus = pd.read_parquet(DATA_PROCESSED / 'corpus_cleaned.parquet')

tweets_etiquetados = pd.read_parquet(DATA_PROCESSED / 'tweets_etiquetados_final.parquet')

tweets_salud = tweets_etiquetados[tweets_etiquetados['etiqueta_salud'] == 1].copy()

df_salud = tweets_salud.merge(
    corpus[['id_doc', 'Texto_limpio']],
    on='id_doc', how='left'
)

print(f'Corpus completo    : {len(corpus):,} tweets')
print(f'Subcorpus de salud : {len(df_salud):,} tweets')
df_salud.head(2)


In [ ]:
# ============================================================
# CELL 2 - CARGAR Y FILTRAR EMBEDDINGS
# ============================================================
embeddings_all = np.load(DATA_PROCESSED / 'tweet_embeddings.npy')
print(f'Embeddings totales  : {embeddings_all.shape}')

mask_salud       = tweets_etiquetados['etiqueta_salud'] == 1
embeddings_salud = embeddings_all[mask_salud.values]

print(f'Embeddings de salud : {embeddings_salud.shape}')
assert len(embeddings_salud) == len(df_salud), (
    f"Mismatch: {len(embeddings_salud)} embeddings vs {len(df_salud)} tweets"
)
print('OK - embeddings alineados correctamente')


In [ ]:
# ============================================================
# CELL 3 - STOPWORDS Y VECTORIZADOR
# ============================================================
nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
spanish_stopwords = list(nlp.Defaults.stop_words)

twitter_sw = [
    'rt', 'http', 'https', 'co', 'amp', 'via', 'q', 'xq', 'x',
    'si', 'ya', 'asi', 'tan', 'ser', 'hay', 'ver', 'hoy',
]
all_stopwords = list(set(spanish_stopwords + twitter_sw))

vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    max_features=5000,
    min_df=5,
    ngram_range=(1, 2),
    strip_accents=None
)

documents = df_salud['Texto_limpio'].fillna('').tolist()
print(f'Documentos para BERTopic: {len(documents):,}')


In [ ]:
# ============================================================
# CELL 4 - FUNCIONES DE COHERENCIA
# ============================================================

def get_topic_words(topic_model, top_n=10):
    topics_words = []
    for topic_id, word_scores in topic_model.get_topics().items():
        if topic_id == -1:
            continue
        words = [word for word, _ in word_scores[:top_n]]
        topics_words.append(words)
    return topics_words


def compute_bertopic_coherence(topic_model, documents, vectorizer_model, top_n_words=10):
    analyzer       = vectorizer_model.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in documents]
    dictionary     = Dictionary(tokenized_docs)
    dictionary.filter_extremes(no_below=5, no_above=0.9)
    corpus_bow     = [dictionary.doc2bow(doc) for doc in tokenized_docs]
    topics_words   = get_topic_words(topic_model, top_n=top_n_words)
    cm = CoherenceModel(
        topics=topics_words, texts=tokenized_docs,
        dictionary=dictionary, corpus=corpus_bow, coherence='c_v'
    )
    return cm.get_coherence(), cm.get_coherence_per_topic()


In [ ]:
# ============================================================
# CELL 5 - BUSQUEDA EN GRILLA DE nr_topics
# ============================================================
results = []

for nr in NR_TOPICS_GRID:
    print(f'\n--- nr_topics={nr} ---')
    tm = BERTopic(
        vectorizer_model=vectorizer_model,
        nr_topics=nr,
        min_topic_size=MIN_TOPIC_SIZE,
        calculate_probabilities=False,
        verbose=False,
        language="spanish"
    )
    topics, _ = tm.fit_transform(documents, embeddings=embeddings_salud)
    new_topics = tm.reduce_outliers(
        documents, topics, strategy="c-tf-idf", embeddings=embeddings_salud
    )
    tm.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

    coherence, _ = compute_bertopic_coherence(tm, documents, vectorizer_model)
    n_outliers   = sum(1 for t in tm.topics_ if t == -1)
    results.append({
        "nr_topics"     : nr,
        "n_topics_final": len([t for t in tm.get_topics() if t != -1]),
        "coherence_c_v" : coherence,
        "pct_outliers"  : n_outliers / len(documents)
    })
    r = results[-1]
    print(f'  Topicos: {r["n_topics_final"]}  Coh: {r["coherence_c_v"]:.4f}  Outliers: {r["pct_outliers"]:.2%}')

df_grid = pd.DataFrame(results)
print()
print(df_grid.sort_values("coherence_c_v", ascending=False).to_string(index=False))


In [ ]:
# ============================================================
# CELL 6 - MODELO FINAL
# ============================================================
topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    nr_topics=NR_TOPICS_FINAL,
    min_topic_size=MIN_TOPIC_SIZE,
    calculate_probabilities=False,
    verbose=True,
    language="spanish"
)

topics, probs = topic_model.fit_transform(documents, embeddings=embeddings_salud)
new_topics    = topic_model.reduce_outliers(
    documents, topics, strategy="c-tf-idf", embeddings=embeddings_salud
)
topic_model.update_topics(documents, topics=new_topics, vectorizer_model=vectorizer_model)

tm_res = topic_model.get_topic_info()
print(tm_res.head(10))


In [ ]:
# ============================================================
# CELL 7 - TABLA DE RESULTADOS FINAL
# ============================================================
coh_global, coh_per_topic = compute_bertopic_coherence(
    topic_model=topic_model,
    documents=documents,
    vectorizer_model=vectorizer_model,
    top_n_words=10
)
print(f'Coherencia global (c_v): {coh_global:.4f}')

topic_ids    = [t for t in topic_model.get_topics() if t != -1]
coherence_df = pd.DataFrame({"Topic": topic_ids, "Coherence_c_v": coh_per_topic})

total_docs  = tm_res["Count"].sum()
top         = tm_res.sort_values("Count", ascending=False).head(20).copy()
top["Keywords"]   = top["Representation"].apply(lambda ws: ", ".join(ws))
top["Porcentaje"] = (top["Count"] / total_docs * 100).round(2)
top = top.merge(coherence_df, on="Topic", how="left")

final_table = top[["Topic", "Count", "Porcentaje", "Coherence_c_v", "Keywords"]]
pd.set_option("display.max_colwidth", None)
print(final_table.to_string(index=False))


In [ ]:
# ============================================================
# CELL 8 - ASIGNAR TOPICO Y GUARDAR
# ============================================================
import pickle

df_salud = df_salud.copy()
df_salud['topico'] = topic_model.topics_

coherence_table = (
    tm_res[tm_res["Topic"] != -1].merge(coherence_df, on="Topic", how="left")
)

ruta_excel = DATA_PROCESSED / 'bertopic_salud_resultados.xlsx'
with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
    final_table.to_excel(writer,     sheet_name='Top_topicos',          index=False)
    coherence_table.to_excel(writer, sheet_name='Coherencia_por_topico',index=False)
    df_grid.to_excel(writer,         sheet_name='Grid_nr_topics',       index=False)
    df_salud[['id_doc', 'topico', 'categoria_detectada']].to_excel(
        writer, sheet_name='Tweets_por_topico', index=False
    )
print('[GUARDADO] bertopic_salud_resultados.xlsx')

df_salud.to_parquet(DATA_PROCESSED / 'salud_tweets_bertopic.parquet', index=False)
print('[GUARDADO] salud_tweets_bertopic.parquet')

print()
print('Notebook 07 completado.')
print('Siguiente -> 08_extraer_frecuencias_POS.ipynb')
